# CR-HSDPA r=0.50 Training - Epochs 1 \u2013 50 (1\u201350 of 200)
**Thesis Topic:** A Modified Attention-PestNet with Channel-Reduced Hierarchical Scaled Dot-Product Attention for Efficient Insect Pest Detection  
**Authors:** Lean Adrian Murillo, James Oliver C. Mendoza, DM Rashid P. Ferrer, Charisse P. Barbosa (University of Mindanao)  
**Target Architecture:** CR-HSDPA (`cfg/cr_hsdpa_050.yaml`, r=0.50)  
**Dataset:** IP102-YOLO (`dmrashidpferrer/ip102-yolo-dataset`)  
**Training Stage:** Epochs 1 \u2013 50 (`lr0=0.01`, SGD, batch=32, imgsz=640, Dual T4)  
**Checkpointing:** `epochs=50` per stage, `save_period=10` + auto `best.pt/last.pt`. Publish stage output as Kaggle dataset for next stage.  
**Branch:** `HSDPA-DM-Testing` (reuses `90377Sednaaa/HSDPA`, account-agnostic via Kaggle CLI).  
---
Run cells top-to-bottom exactly once (Rule 4: single `model.train()` cell). After run, save output version and attach it to the next stage kernel.


In [ ]:
# 1. Setup repository and install ultralytics in editable mode
# Works on any Kaggle account (authenticated via Kaggle CLI) and any GitHub checkout.
import os
os.environ["WANDB_MODE"] = "disabled"
import sys

REPO_URL = "https://github.com/90377Sednaaa/HSDPA.git"
BRANCH = "HSDPA-DM-Testing"
REPO_DIR = "/kaggle/working/HSDPA"
if not os.path.exists(REPO_DIR):
    !git clone -b {BRANCH} {REPO_URL} {REPO_DIR}
else:
    %cd {REPO_DIR}
    !git fetch origin
    !git checkout {BRANCH}
    !git pull origin {BRANCH}

%cd {REPO_DIR}
!pip install -q -e .

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

print("Environment ready at:", REPO_DIR, "| branch:", BRANCH)


In [ ]:
# 2. Check GPU environment and CUDA capabilities
import torch
from ultralytics import YOLO

print("PyTorch version:", torch.__version__)
print("CUDA Available:", torch.cuda.is_available())
device_count = torch.cuda.device_count()
print("GPU Count:", device_count)
for i in range(device_count):
    print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")


In [ ]:
# 3. Verify IP102-YOLO dataset directory and images
# Attach in Kaggle UI or CLI: dmrashidpferrer/ip102-yolo-dataset
import os, glob, yaml

DATA_ROOT = None
for root, dirs, files in os.walk("/kaggle/input"):
    if "images" in dirs and os.path.exists(os.path.join(root, "images/train")):
        DATA_ROOT = root
        break

if not DATA_ROOT:
    raise FileNotFoundError("Could not locate IP102 dataset in /kaggle/input. Attach dmrashidpferrer/ip102-yolo-dataset.")

print(f"Dataset root verified at: {DATA_ROOT}")
train_count = len(os.listdir(os.path.join(DATA_ROOT, "images/train")))
val_count = len(os.listdir(os.path.join(DATA_ROOT, "images/val")))
print(f"Train images: {train_count} (Expected: 15,178)")
print(f"Validation images: {val_count} (Expected: 3,798)")

with open("cfg/ip102.yaml", "r") as f:
    ip102_cfg = yaml.safe_load(f)
ip102_cfg["path"] = DATA_ROOT
with open("cfg/ip102.yaml", "w") as f:
    yaml.dump(ip102_cfg, f, default_flow_style=False, sort_keys=False)
print("cfg/ip102.yaml dynamically configured and ready!")


In [ ]:
# 4. Train CR-HSDPA r=0.50 (Stage 1: 50 epochs from scratch)
import torch
import os, glob, time
os.environ["WANDB_MODE"] = "disabled"
from ultralytics import YOLO

MODEL_CFG = "cfg/cr_hsdpa_050.yaml"
PROJECT_NAME = "dataset_r050_stage1"
EXP_NAME = "ip102_cr050_stage1"

# Resume only if THIS stage was interrupted (same project/name -> last.pt)
resume_ckpts = glob.glob(f"{PROJECT_NAME}/{EXP_NAME}/weights/last.pt")
if resume_ckpts:
    resume_ckpt = resume_ckpts[0]
    print(f"Found interrupted Stage 1 checkpoint, resuming: {resume_ckpt}")
    model = YOLO(resume_ckpt)
    resume_flag = True
else:
    print(f"Initializing fresh CR-HSDPA r=0.50 model from {MODEL_CFG}")
    model = YOLO(MODEL_CFG)
    resume_flag = False

# Multi-GPU support for dual T4
device_arg = [0, 1] if torch.cuda.device_count() >= 2 else 0

train_start_wall = time.time()

results = model.train(
    data="cfg/ip102.yaml",
    epochs=50,
    batch=32,
    imgsz=640,
    device=device_arg,
    optimizer="SGD",
    lr0=0.01,
    momentum=0.9,
    amp=True,
    save_period=10,
    project=PROJECT_NAME,
    name=EXP_NAME,
    exist_ok=True,
    workers=4,
    resume=resume_flag,
    verbose=True
)

train_elapsed_sec = time.time() - train_start_wall
train_elapsed_hrs = train_elapsed_sec / 3600.0
print(f"\nStage 1 CR-HSDPA r=0.50 training completed in {train_elapsed_hrs:.2f} hours ({train_elapsed_sec:.1f} s).")
os.makedirs(f"{PROJECT_NAME}/{EXP_NAME}", exist_ok=True)
with open(f"{PROJECT_NAME}/{EXP_NAME}/training_time.txt", "w") as f:
    f.write(f"{train_elapsed_sec:.2f}\n")


In [ ]:
# 5. Comprehensive Thesis Evaluation & Efficiency Benchmark (Section 2.7 Protocol)
import time, os, glob
import torch
import pandas as pd
from ultralytics import YOLO

best_ckpt = f"{PROJECT_NAME}/{EXP_NAME}/weights/best.pt"
if os.path.exists(best_ckpt):
    print(f"=== Evaluating CR-HSDPA r=0.50 Stage 1 Best Checkpoint: {best_ckpt} ===\n")
    eval_model = YOLO(best_ckpt)

    # 1. Detection Performance (mAP@50, mAP@50:95, Precision, Recall)
    val_results = eval_model.val(data="cfg/ip102.yaml", split="val", batch=32, imgsz=640, device=0)
    p = val_results.box.mp
    r = val_results.box.mr
    map50 = val_results.box.map50
    map50_95 = val_results.box.map

    # 2. Computational Efficiency Benchmark (Section 2.7: FP32, batch 1, 20 warmup, 200 timed runs)
    device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    bench_model = eval_model.model.to(device).float()
    bench_model.eval()
    dummy_input = torch.randn(1, 3, 640, 640, device=device)

    with torch.no_grad():
        for _ in range(20):
            _ = bench_model(dummy_input)
        if torch.cuda.is_available():
            torch.cuda.synchronize()

        start_time = time.perf_counter()
        for _ in range(200):
            _ = bench_model(dummy_input)
            if torch.cuda.is_available():
                torch.cuda.synchronize()
        total_time = time.perf_counter() - start_time

    latency_ms = (total_time / 200.0) * 1000.0
    fps = 200.0 / total_time
    total_params = sum(x.numel() for x in eval_model.model.parameters())
    model_size_mb = os.path.getsize(best_ckpt) / (1024 * 1024)

    time_file = f"{PROJECT_NAME}/{EXP_NAME}/training_time.txt"
    train_time_hrs = None
    if os.path.exists(time_file):
        with open(time_file, "r") as f:
            train_time_hrs = float(f.read().strip()) / 3600.0

    print("\n" + "=" * 65)
    print("   THESIS EVALUATION METRICS SUMMARY - CR-HSDPA r=0.50 STAGE 1 (Section 2.7)")
    print("=" * 65)
    print(f"  Precision (P):          {{p:.4f}} ({{p*100:.2f}}%)")
    print(f"  Recall (R):             {{r:.4f}} ({{r*100:.2f}}%)")
    print(f"  mAP@50:                 {{map50:.4f}} ({{map50*100:.2f}}%)")
    print(f"  mAP@50:95:              {{map50_95:.4f}} ({{map50_95*100:.2f}}%)")
    print(f"  Parameters:             {{total_params:,}}")
    if train_time_hrs:
        print(f"  Training Time:          {{train_time_hrs:.2f}} hours (Stage 1)")
    print(f"  Inference Latency:      {{latency_ms:.2f}} ms")
    print(f"  Throughput (FPS):       {{fps:.2f}} FPS")
    print(f"  Model File Size:        {{model_size_mb:.2f}} MB")
    print("=" * 65)

    summary_df = pd.DataFrame([{{
        "Model": "CR-HSDPA r=0.50 (Stage 1)",
        "Precision": p,
        "Recall": r,
        "mAP@50": map50,
        "mAP@50-95": map50_95,
        "Parameters": total_params,
        "Training_Time_Hours": round(train_time_hrs, 2) if train_time_hrs else "N/A",
        "Latency_ms": latency_ms,
        "FPS": fps,
        "ModelSize_MB": model_size_mb
    }}])
    out_csv = f"{PROJECT_NAME}/{EXP_NAME}/thesis_metrics_summary_cr050_stage1.csv"
    summary_df.to_csv(out_csv, index=False)
    print(f"Saved Stage 1 metrics summary to: {{out_csv}}")
else:
    print(f"Checkpoint {{best_ckpt}} not found.")
